# ABSA — Model Training & Comparison (Colab)

Trains and compares both ABSA stages, selects a final model, and exports it.

**Runtime → Change runtime type → T4 GPU** before running. The notebook works on
CPU too (the baselines take seconds; DistilBERT takes ~45 min), but DeBERTa is
only practical on a GPU.

As with `01_eda.ipynb`, **the logic is not in this notebook** — it lives in
`ml/training/`, `ml/evaluation/` and `scripts/`, which this notebook drives.
That keeps training reproducible from the CLI and diffable in git.

## Protocol

* Train on `train`, tune on `dev`, touch `test` **once** at the end.
* **Macro F1** selects the sentiment model; **micro F1** selects aspect detection.
  Accuracy is never the criterion — `neutral` is 5.3% of pairs, so a model that
  refuses to predict it still scores ~95% on that class.
* Splits are grouped by review; the build asserts no leakage.

## 1. Environment

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_NAME = "absa-platform"
REPO_URL = os.environ.get("ABSA_REPO_URL", "https://github.com/tushpatil02/absa-platform.git")

if IN_COLAB:
    if not Path(REPO_NAME).exists():
        !git clone -q $REPO_URL $REPO_NAME
    else:
        !cd $REPO_NAME && git pull -q
    REPO_ROOT = Path(REPO_NAME).resolve()
    !pip install -q transformers
else:
    REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

import torch

HAS_GPU = torch.cuda.is_available()
print(f"Repo:   {REPO_ROOT}")
print(f"torch:  {torch.__version__}")
print(f"GPU:    {torch.cuda.get_device_name(0) if HAS_GPU else 'none — CPU fallback'}")
if not HAS_GPU:
    print("\nNo GPU. Baselines still run in seconds; use --model distilbert-base-uncased")
    print("for the transformer, and expect ~45 min per stage.")

## 2. Data

Skip if `01_eda.ipynb` already built it in this runtime.

In [ ]:
if not Path("data/processed/asc_train.csv").exists():
    !python scripts/download_data.py
    !python scripts/build_dataset.py
else:
    print("data/processed already built")

## 3. Baselines first

TF-IDF (word + char n-grams) with logistic regression and a calibrated linear
SVM. These set the bar a transformer has to clear to justify its cost.

In [ ]:
!python scripts/train_baseline.py

### Why not accuracy

Look at the two Stage B baselines above. The SVM reports **higher accuracy** than
logistic regression while scoring **0.000 F1 on `neutral`** — it never predicts
that class at all. Selecting on accuracy would have picked the model that
silently dropped a third of the label space.

## 4. Transformers

* **Stage A** — 12 sigmoid outputs, `BCEWithLogitsLoss` with per-label
  `pos_weight` so rare aspects are not ignored.
* **Stage B** — sentence pair `[CLS] review [SEP] aspect description [SEP]`, so
  the model can attend to *which* aspect it is being asked about. Class-weighted
  cross-entropy for the neutral imbalance.

`deberta-v3-base` needs `sentencepiece`; DistilBERT does not.

In [ ]:
MODEL = "microsoft/deberta-v3-base" if HAS_GPU else "distilbert-base-uncased"
EPOCHS = 4 if HAS_GPU else 3
BATCH = 16

if HAS_GPU:
    !pip install -q sentencepiece
print(f"model={MODEL}  epochs={EPOCHS}  batch={BATCH}")

In [ ]:
!python scripts/train_transformer.py --stage asc --model $MODEL --epochs $EPOCHS --batch-size $BATCH

In [ ]:
!python scripts/train_transformer.py --stage acd --model $MODEL --epochs $EPOCHS --batch-size $BATCH

## 5. Compare and select

Builds the comparison table from every `models/metadata/*_results.json` written
so far, and reports the aspect-conditioning diagnostic.

In [ ]:
!python scripts/compare_models.py

### The diagnostic that matters

Overall accuracy hides whether the model conditions on the aspect at all. Most
reviews are uniformly positive or negative, so a model that reads only overall
tone still scores well.

`scripts/compare_models.py` therefore also reports accuracy on **mixed reviews** —
those carrying different polarities for different aspects, e.g. *"camera is
excellent, but the battery drains quickly"*. On those, a tone-reading model is
wrong by construction on half the aspects.

The TF-IDF baseline scores **0.545** on mixed reviews versus **0.840** on uniform
ones, and gives **75.5%** of mixed reviews a single polarity for every aspect.
That gap is the strongest argument for the transformer.

## 6. Test inference on the exported artefacts

In [ ]:
from ml.inference.predictor import load_predictor
from ml.preprocessing.transform import load_taxonomy

taxonomy = load_taxonomy(Path("ml/config/aspect_taxonomy.yaml"))
predictor = load_predictor(Path("models"), taxonomy, prefer="auto")
print(f"loaded: {predictor.model_name}\n")

for review in [
    "The display is beautiful and the camera takes excellent photos, but the battery life is disappointing.",
    "Fast delivery, well packaged. The phone itself is terribly slow though.",
]:
    result = predictor.analyze(review)
    print(review)
    for prediction in result.aspects:
        sentiment = prediction.sentiment
        print(
            f"   {prediction.display_name:<22} {sentiment.polarity:<9}"
            f" {sentiment.score:>5}/10  conf {sentiment.confidence:.2f}"
        )
    print()

## 7. Push to the Hugging Face Hub

Trained weights do not belong in git — a portfolio repo with 500 MB of
`.safetensors` is both unwieldy and past GitHub's limits. They go to the Hub,
and the backend pulls them at build time.

Add `HF_TOKEN` via the Colab **secrets** panel (key icon in the left sidebar).
Never paste a token into a cell.

In [ ]:
HF_USERNAME = ""  # e.g. "tusharpatil" — leave blank to skip this step

if HF_USERNAME:
    token = None
    if IN_COLAB:
        from google.colab import userdata

        try:
            token = userdata.get("HF_TOKEN")
        except Exception:
            print("Add HF_TOKEN in the Colab secrets panel (key icon) first.")
    else:
        token = os.environ.get("HF_TOKEN")

    if token:
        !pip install -q huggingface_hub
        from huggingface_hub import HfApi

        api = HfApi(token=token)
        for local, repo in [
            ("models/aspect_detector", f"{HF_USERNAME}/absa-aspect-detector"),
            ("models/sentiment_classifier", f"{HF_USERNAME}/absa-sentiment-classifier"),
        ]:
            if Path(local).exists():
                api.create_repo(repo, exist_ok=True)
                api.upload_folder(folder_path=local, repo_id=repo)
                print(f"pushed {local} -> https://huggingface.co/{repo}")
else:
    print("HF_USERNAME not set — skipping upload.")

## 8. Next

Commit `models/metadata/*.json` (small, and the source for the README's results
table) — the weights stay out of git via `.gitignore`.

Then run the API against the exported artefacts:

```bash
cd backend && uvicorn app.main:app --reload
```